# Robust04 Final Project — Research Narrative + Ablations

This notebook is written as a **class project presentation / research report**:

- What we tried (and why)
- What worked / didn’t work
- Ablations + comparisons using metrics
- How to reproduce the final run files

## Problem statement
Given Robust04 queries and qrels (50 judged queries), produce **three different runs** (`run_1`, `run_2`, `run_3`) and optimize effectiveness on the judged split.

## Constraints (course-style)
- Use the provided query file (`Files-20260104/queriesROBUST.txt`). In this dataset, each qid has **a single title string**.
- Ensure the approach is reproducible.
- Include at least 3 different retrieval methods, including at least one neural method.

## Evaluation protocol
- Tune on the **first 50 judged queries** (qids 301–350) using **MAP**.
- Also report retrieval-stage metrics (**Recall@K**, **nDCG@K**) to justify candidate-set design.

## Key results (from this project’s exploration)
- Tuned RM3 baseline: MAP ≈ 0.2719
- Best retrieval-stage fusion (no reranking): MAP ≈ 0.2997
- Passage-level MonoT5 reranking can substantially improve MAP beyond fusion.

The rest of the notebook explains the research flow that led to these decisions, with runnable evaluation code.

## Presentation roadmap + research questions

### Research questions
1. **How strong is a tuned lexical baseline on Robust04?**
2. **Do neural retrievers help retrieval-stage effectiveness or recall?**
3. **Does fusing multiple retrievers justify its added complexity?**
4. **Given a strong candidate set, does neural reranking yield further gains (especially for long documents)?**

### Chronological research flow (decision log)
This is the order we explored the system, and the reason for each step:

1. **Start from a strong baseline (BM25 + RM3).**
   - Motivation: Robust04 is newswire; title queries are keyword-like.
   - Outcome: establishes the baseline MAP level.

2. **Add neural retrieval methods as candidate generators (SPLADE++, SPLADE-v3, dense BGE).**
   - Motivation: handle lexical mismatch / synonymy.
   - Outcome: some methods improve recall or ranking on different query types.

3. **Fuse retrievers to improve candidate set quality/coverage.**
   - Motivation: retrievers are complementary; reranking can only reorder retrieved docs.
   - Main baseline: min-max score fusion + weighted sum.
   - Key alternative: **RRF rank fusion**.

4. **Apply neural reranking on top of the fused candidate set.**
   - Motivation: fused retrieval improves candidate set, but relevance modeling is still shallow.
   - Robust04 docs are long → we use **passage-level** reranking + MaxP aggregation.

5. **Record negative results** (methods we tried that didn’t surpass the baseline).
   - Motivation: demonstrate breadth of exploration and justify the final design.

### What you should take from this notebook
- A reproducible pipeline (evaluation + run generation).
- A clear ablation-style story: each added component is motivated and justified with metrics.

In [ ]:
import os
import math
import re
import time
from collections import defaultdict
from pathlib import Path
from typing import Dict, Iterable, List, Tuple

# Prevent Lucene memory-segment issues in some environments
os.environ.setdefault(
    'JAVA_TOOL_OPTIONS',
    '-Dorg.apache.lucene.store.MMapDirectory.enableMemorySegments=false',
)

import torch

from pyserini.encode import SpladeQueryEncoder
from pyserini.search.lucene import LuceneHnswDenseSearcher, LuceneImpactSearcher, LuceneSearcher

print('torch:', torch.__version__)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)


## Data and evaluation setup

### Data
- **Queries**: `Files-20260104/queriesROBUST.txt` (249 queries)
- Each query is **one title string per qid** (no description/narrative fields in the course data).
- **Judged/tuning split**: first 50 qids (301–350)
- **Qrels**: `Files-20260104/qrels_50_Queries`

### Metrics
We use a mix of *optimization* and *diagnostic* metrics:

- **MAP**: primary tuning metric on the 50 judged queries (standard TREC-style metric).
- **Recall@K**: measures whether our candidate set contains relevant documents.
- **nDCG@K**: measures ranking quality at depth K (Järvelin & Kekäläinen, 2002).

Why report Recall@K and nDCG@K?
- If recall is low at K=1000, reranking cannot fix it (relevant docs are missing).
- If recall is high but nDCG is low, reranking has room to improve ordering.

In [ ]:
QUERIES_PATH = Path('Files-20260104/queriesROBUST.txt')
QRELS_PATH = Path('Files-20260104/qrels_50_Queries')

def read_queries_tsv(path: Path) -> Dict[str, str]:
    queries: Dict[str, str] = {}
    for line in path.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if not line:
            continue
        qid, query = line.split('	', 1)
        queries[qid] = query
    return queries

def read_qrels(path: Path) -> Dict[str, Dict[str, int]]:
    qrels: Dict[str, Dict[str, int]] = defaultdict(dict)
    for line in path.read_text(encoding='utf-8').splitlines():
        parts = line.strip().split()
        if len(parts) != 4:
            continue
        qid, _, docid, rel = parts
        qrels[qid][docid] = int(rel)
    return qrels

def average_precision(docids: List[str], rels: Dict[str, int]) -> float:
    num_rel = sum(1 for r in rels.values() if r > 0)
    if num_rel == 0:
        return 0.0
    hit = 0
    s = 0.0
    for i, d in enumerate(docids, start=1):
        if rels.get(d, 0) > 0:
            hit += 1
            s += hit / i
    return s / num_rel

def mean_ap(run: Dict[str, List[str]], qrels: Dict[str, Dict[str, int]]) -> float:
    return sum(average_precision(run[qid], qrels[qid]) for qid in run) / float(len(run))

def recall_at_k(docids: List[str], rels: Dict[str, int], k: int) -> float:
    num_rel = sum(1 for r in rels.values() if r > 0)
    if num_rel == 0:
        return 0.0
    seen = 0
    for d in docids[: max(0, int(k))]:
        if rels.get(d, 0) > 0:
            seen += 1
    return float(seen) / float(num_rel)

def _dcg_at_k(docids: List[str], rels: Dict[str, int], k: int) -> float:
    k = max(0, int(k))
    s = 0.0
    for i, d in enumerate(docids[:k], start=1):
        rel = int(rels.get(d, 0))
        if rel <= 0:
            continue
        gain = (2.0 ** rel) - 1.0
        s += gain / math.log2(i + 1.0)
    return float(s)

def ndcg_at_k(docids: List[str], rels: Dict[str, int], k: int) -> float:
    k = max(0, int(k))
    dcg = _dcg_at_k(docids, rels, k)
    if dcg <= 0.0:
        return 0.0
    ideal_rels = sorted((int(r) for r in rels.values() if int(r) > 0), reverse=True)
    if not ideal_rels:
        return 0.0
    ideal_docids = [str(i) for i in range(len(ideal_rels))]
    ideal_map = {str(i): int(ideal_rels[i]) for i in range(len(ideal_rels))}
    idcg = _dcg_at_k(ideal_docids, ideal_map, k)
    if idcg <= 0.0:
        return 0.0
    return float(dcg) / float(idcg)

def mean_recall_at_k(run: Dict[str, List[str]], qrels: Dict[str, Dict[str, int]], k: int) -> float:
    return sum(recall_at_k(run[qid], qrels[qid], k) for qid in run) / float(len(run))

def mean_ndcg_at_k(run: Dict[str, List[str]], qrels: Dict[str, Dict[str, int]], k: int) -> float:
    return sum(ndcg_at_k(run[qid], qrels[qid], k) for qid in run) / float(len(run))

all_queries = read_queries_tsv(QUERIES_PATH)
train_qids = list(all_queries.keys())[:50]
train_queries = {qid: all_queries[qid] for qid in train_qids}
qrels = read_qrels(QRELS_PATH)

print('queries total:', len(all_queries))
print('train queries:', len(train_queries))
print('qrels qids:', len(qrels))


## Retrieval methods: what they are and why we used them

Robust04 is a classic ad-hoc newswire retrieval benchmark. The course-provided queries here are **title-only**, so a lot of the signal is lexical, but neural retrieval/reranking can still help with synonymy and semantic mismatch.

Below are the *retrieval-stage* methods we used to build candidate sets.

### 1) BM25 + RM3 (lexical baseline)
What it is:
- **BM25** ranks documents by term matching with length normalization (Robertson & Zaragoza, 2009).
- **RM3** is pseudo-relevance feedback (PRF): expand the query using terms from top-ranked documents (Lavrenko & Croft, 2001; Abdul-Jaleel et al., 2004).

Why it’s a good baseline here:
- Robust04 queries often have strong keyword intent.
- RM3 often boosts recall by adding related terms.

What we tuned:
- BM25: `k1=0.9`, `b=0.4`
- RM3: `fb_terms=20`, `fb_docs=5`, `oqw=0.5`

### 2) SPLADE++ and SPLADE-v3 (learned sparse retrieval)
What it is:
- SPLADE models learn sparse lexical expansion with transformer-based representations (Formal et al., 2021).
- Retrieval remains inverted-index-based (fast), but matching can become more semantic.

Why it helps:
- Captures synonymy/relatedness while still using exact term evidence.
- Often improves recall compared to purely lexical runs.

Why we used two variants:
- SPLADE++ and SPLADE-v3 differ in training recipes/capacity; on some queries one can outperform the other.

### 3) Dense retrieval (BGE embeddings)
What it is:
- Dual-encoder style dense retrieval: encode queries/docs into vectors and retrieve nearest neighbors (e.g., DPR-style setup: Karpukhin et al., 2020).
- We use an ANN index (HNSW; Malkov & Yashunin, 2018).

Why it helps:
- Retrieves semantically related documents even when exact term overlap is weak.

Important constraint in this project:
- Dense / impact indexes generally do not store raw text, so we fetch text from the `robust04` Lucene index when reranking.

In [ ]:
def retrieve_run(searcher, queries: Dict[str, str], k: int = 1000) -> Dict[str, List[str]]:
    run: Dict[str, List[str]] = {}
    for qid, query in queries.items():
        hits = searcher.search(query, k=k)
        run[qid] = [h.docid for h in hits]
    return run

k = 1000

t0 = time.time()
rm3 = LuceneSearcher.from_prebuilt_index('robust04')
rm3.set_bm25(0.9, 0.4)
rm3.set_rm3(20, 5, 0.5)

spladepp_encoder = SpladeQueryEncoder('naver/splade-cocondenser-ensembledistil', device=device)
spladepp = LuceneImpactSearcher.from_prebuilt_index('beir-v1.0.0-robust04.splade-pp-ed', spladepp_encoder)

spladev3_encoder = SpladeQueryEncoder('naver/splade-v3-distilbert', device=device)
spladev3 = LuceneImpactSearcher.from_prebuilt_index('beir-v1.0.0-robust04.splade-v3', spladev3_encoder)

dense = LuceneHnswDenseSearcher.from_prebuilt_index(
    'beir-v1.0.0-robust04.bge-base-en-v1.5.hnsw',
    ef_search=1000,
    encoder='BgeBaseEn15',
)

try:
    rm3_run = retrieve_run(rm3, train_queries, k=k)
    spladepp_run = retrieve_run(spladepp, train_queries, k=k)
    spladev3_run = retrieve_run(spladev3, train_queries, k=k)
    dense_run = retrieve_run(dense, train_queries, k=k)
finally:
    rm3.close(); spladepp.close(); spladev3.close(); dense.close()

print('elapsed_sec', round(time.time() - t0, 1))

print('MAP (RM3):', f'{mean_ap(rm3_run, qrels):.4f}')
print('MAP (SPLADE++):', f'{mean_ap(spladepp_run, qrels):.4f}')
print('MAP (SPLADE-v3):', f'{mean_ap(spladev3_run, qrels):.4f}')
print('MAP (Dense BGE):', f'{mean_ap(dense_run, qrels):.4f}')


## Retrieval-stage diagnostic metrics: Recall@K and nDCG@K

In a 2-stage system (retrieve → rerank), retrieval-stage metrics answer:

- **Recall@K**: do we *retrieve* the relevant documents at all?
- **nDCG@K**: do we rank relevant documents near the top?

How we interpret them:
- If **Recall@1000** is low, the candidate generator is missing relevant docs.
- If Recall@1000 is decent but **nDCG@10** is low, reranking may yield gains.

We compute these for every retrieval method and every fusion method we consider, so we can justify added complexity with numbers.

In [ ]:
eval_ks = [10, 20, 50, 100, 200, 500, 1000]

runs = {
    'RM3': rm3_run,
    'SPLADE++': spladepp_run,
    'SPLADE-v3': spladev3_run,
    'Dense(BGE)': dense_run,
}

for name, run in runs.items():
    rec = {k: mean_recall_at_k(run, qrels, k) for k in eval_ks}
    nd = {k: mean_ndcg_at_k(run, qrels, k) for k in eval_ks}
    rec_s = ' '.join(f'{k}:{rec[k]:.4f}' for k in eval_ks)
    nd_s = ' '.join(f'{k}:{nd[k]:.4f}' for k in eval_ks)
    print(name, 'Recall@K', rec_s)
    print(name, 'nDCG@K', nd_s)
    print()


## Ablation 1 — Single retrievers (candidate generators)

**Question:** If we only used one retriever, which one is best? And what failure modes does each have?

We compare four candidate generators:
- RM3 (lexical PRF)
- SPLADE++ (learned sparse)
- SPLADE-v3 (learned sparse)
- Dense BGE (dense retrieval)

**Metrics reported:**
- **MAP** (overall effectiveness on judged queries)
- **Recall@1000** (candidate coverage)
- **nDCG@10** (top-ranking quality)

Interpretation guide:
- High Recall@1000 but lower nDCG@10 suggests reranking might help.
- Low Recall@1000 suggests the retriever is missing relevant documents and needs fusion or a stronger retrieval model.

In [ ]:
def summarize_run_row(name: str, run: Dict[str, List[str]], qrels: Dict[str, Dict[str, int]]):
    return {
        'name': name,
        'MAP': mean_ap(run, qrels),
        'Recall@1000': mean_recall_at_k(run, qrels, 1000),
        'nDCG@10': mean_ndcg_at_k(run, qrels, 10),
    }


def print_table(rows: List[Dict[str, float]]):
    cols = ['name', 'MAP', 'Recall@1000', 'nDCG@10']
    widths = {c: len(c) for c in cols}
    for r in rows:
        widths['name'] = max(widths['name'], len(str(r['name'])))
        for c in cols[1:]:
            widths[c] = max(widths[c], len(f"{r[c]:.4f}"))

    header = '  '.join(c.ljust(widths[c]) for c in cols)
    print(header)
    print('-' * len(header))
    for r in rows:
        parts = [str(r['name']).ljust(widths['name'])]
        for c in cols[1:]:
            parts.append(f"{r[c]:.4f}".ljust(widths[c]))
        print('  '.join(parts))


rows = [
    summarize_run_row('RM3', rm3_run, qrels),
    summarize_run_row('SPLADE++', spladepp_run, qrels),
    summarize_run_row('SPLADE-v3', spladev3_run, qrels),
    summarize_run_row('Dense(BGE)', dense_run, qrels),
]
print_table(rows)

## Fusion: why combine multiple retrievers?

Fusion is our way to build a *stronger candidate set* by combining retrievers that succeed on different query types.

### Why fuse at all?
A reranker can only reorder what it sees. If a relevant document is not retrieved in the top-`K` candidates, reranking cannot recover it.
So we aim to maximize **candidate quality + coverage** first.

### Two broad families of fusion

1. **Score-based fusion** (what we used for `run_2` and `run_3`):
   - Combine retrievers using their similarity scores.
   - Requires score calibration/normalization.

2. **Rank-based fusion** (important alternative):
   - Combine retrievers using only ranks, e.g. **RRF** (Cormack et al., 2009).
   - Robust to score-scale mismatch.

### Why min-max normalization (per query)?
Each retriever’s score distribution is on a different scale.
If we sum raw scores directly, the largest-scale retriever dominates *even if it is worse*.

We chose **per-query min-max normalization** because it is:
- **Simple**: maps scores into `[0, 1]` per query.
- **Scale-robust**: prevents score-unit mismatch.
- **Reproducible**: no learning/calibration stage.

Caveat:
- Min-max can be sensitive to outliers, but empirically it worked well in our judged-query tuning.

### Why a weighted sum?
After normalization, a **weighted sum** is a simple way to express “how much we trust each retriever”.

How weights were chosen in this project:
- Iterative small sweeps / adjustments on the 50 judged queries.
- RM3 kept the highest weight because it is very strong on Robust04.
- SPLADE variants and dense BGE get smaller weights as complementary signals.

Next we compute the min-max weighted fusion baselines (`run_2`, `run_3`) and then compare them to **RRF**.

In [ ]:
def minmax_norm(scores_dict: Dict[str, float]) -> Dict[str, float]:
    if not scores_dict:
        return {}
    vals = list(scores_dict.values())
    mn, mx = min(vals), max(vals)
    if mx - mn < 1e-9:
        return {d: 0.0 for d in scores_dict}
    return {d: (s - mn) / (mx - mn) for d, s in scores_dict.items()}


def retrieve_scores(searcher, query: str, k: int = 1000) -> Dict[str, float]:
    hits = searcher.search(query, k=k)
    return {h.docid: float(h.score) for h in hits}


def fuse_weighted_minmax(
    runs_scores: List[Dict[str, float]],
    weights: List[float],
    depth: int = 1000,
) -> List[str]:
    norms = [minmax_norm(rs) for rs in runs_scores]
    docs = set()
    for n in norms:
        docs |= set(n.keys())

    fused_scores: Dict[str, float] = {}
    for d in docs:
        fused_scores[d] = float(sum(float(w) * n.get(d, 0.0) for w, n in zip(weights, norms)))

    ranked = sorted(fused_scores.items(), key=lambda x: (-x[1], x[0]))
    return [d for d, _ in ranked[:depth]]


run_2_weights = [0.60, 0.25, 0.15]          # rm3, splade++, dense
run_3_weights = [0.55, 0.10, 0.15, 0.20]    # rm3, splade++, splade-v3, dense

rm3 = LuceneSearcher.from_prebuilt_index('robust04')
rm3.set_bm25(0.9, 0.4)
rm3.set_rm3(20, 5, 0.5)

spladepp_encoder = SpladeQueryEncoder('naver/splade-cocondenser-ensembledistil', device=device)
spladepp = LuceneImpactSearcher.from_prebuilt_index('beir-v1.0.0-robust04.splade-pp-ed', spladepp_encoder)

spladev3_encoder = SpladeQueryEncoder('naver/splade-v3-distilbert', device=device)
spladev3 = LuceneImpactSearcher.from_prebuilt_index('beir-v1.0.0-robust04.splade-v3', spladev3_encoder)

dense = LuceneHnswDenseSearcher.from_prebuilt_index(
    'beir-v1.0.0-robust04.bge-base-en-v1.5.hnsw',
    ef_search=1000,
    encoder='BgeBaseEn15',
)

run2: Dict[str, List[str]] = {}
run3: Dict[str, List[str]] = {}

t0 = time.time()
try:
    for i, (qid, query) in enumerate(train_queries.items(), start=1):
        rm3_s = retrieve_scores(rm3, query, k=k)
        pp_s = retrieve_scores(spladepp, query, k=k)
        v3_s = retrieve_scores(spladev3, query, k=k)
        dense_s = retrieve_scores(dense, query, k=k)

        run2[qid] = fuse_weighted_minmax([rm3_s, pp_s, dense_s], run_2_weights, depth=k)
        run3[qid] = fuse_weighted_minmax([rm3_s, pp_s, v3_s, dense_s], run_3_weights, depth=k)

        if i % 10 == 0:
            print('fused', i, '/', len(train_queries))
finally:
    rm3.close(); spladepp.close(); spladev3.close(); dense.close()

print('elapsed_sec', round(time.time() - t0, 1))
print('MAP (fusion run_2):', f'{mean_ap(run2, qrels):.4f}')
print('MAP (fusion run_3):', f'{mean_ap(run3, qrels):.4f}')

In [ ]:
for name, run in [('fusion run_2', run2), ('fusion run_3', run3)]:
    rec = {k: mean_recall_at_k(run, qrels, k) for k in eval_ks}
    nd = {k: mean_ndcg_at_k(run, qrels, k) for k in eval_ks}
    rec_s = ' '.join(f'{k}:{rec[k]:.4f}' for k in eval_ks)
    nd_s = ' '.join(f'{k}:{nd[k]:.4f}' for k in eval_ks)
    print(name, 'Recall@K', rec_s)
    print(name, 'nDCG@K', nd_s)
    print()


## Fusion alternative: RRF (Reciprocal Rank Fusion)

RRF is a rank-based fusion method that does **not** use raw scores (Cormack et al., 2009).

### Why consider RRF?
- Each retriever’s scores live on different scales and are not calibrated.
- RRF only uses ranks, so it is robust to score-scale mismatch.

### RRF formula
If a document has rank `r` (1 is best) in a run, RRF assigns it:

`1 / (k + r)`

and then **sums** this across runs. The hyperparameter `k` (commonly 60) controls how quickly contributions decay.

Below we compute RRF fusions over the same component sets as `run_2` and `run_3`, then compare **MAP**, **Recall@K**, and **nDCG@K** on the judged queries.

In [ ]:
def rrf_fuse(runs: List[List[str]], depth: int, rrf_k: int = 60) -> List[str]:
    scores: Dict[str, float] = {}
    for run in runs:
        for rank, docid in enumerate(run, start=1):
            scores[docid] = float(scores.get(docid, 0.0)) + (1.0 / float(rrf_k + rank))
    ranked = sorted(scores.items(), key=lambda x: (-x[1], x[0]))
    return [d for d, _ in ranked[:depth]]


rrf_k = 60
rrf_run2 = {
    qid: rrf_fuse([rm3_run[qid], spladepp_run[qid], dense_run[qid]], depth=1000, rrf_k=rrf_k)
    for qid in train_queries
}
rrf_run3 = {
    qid: rrf_fuse([rm3_run[qid], spladepp_run[qid], spladev3_run[qid], dense_run[qid]], depth=1000, rrf_k=rrf_k)
    for qid in train_queries
}

print('MAP (RRF run_2):', f'{mean_ap(rrf_run2, qrels):.4f}')
print('MAP (RRF run_3):', f'{mean_ap(rrf_run3, qrels):.4f}')

## Reranking: why MonoT5?

### MonoT5 as a reranker
MonoT5 is a sequence-to-sequence reranker that scores relevance via the model’s preference for generating `true` vs `false` (Nogueira et al., 2020; arXiv:2003.06713).

We format inputs as:

`Query: ... Document: ... Relevant:`

### Why passage-level reranking (MaxP)?
Robust04 documents can be long. A single 512-token truncation often misses the relevant span.

Passage-level reranking is a standard strategy for long-document ranking (see e.g., PARADE: Li et al., 2020; arXiv:2008.09093):
- split each document into overlapping passages
- score each passage
- aggregate back to a document score

We use **MaxP** aggregation (take the maximum passage score), which is effective when relevance evidence may appear anywhere in the document.

## Ablation 2 — Fusion methods (min-max vs RRF)

**Question:** Is fusion just extra complexity, or does it meaningfully improve retrieval-stage effectiveness?

Here we compare:
- Score-fusion baselines: **min-max normalization + weighted sum**
- Rank-fusion alternative: **RRF**

We report the same presentation metrics:
- **MAP** (tuning metric)
- **Recall@1000** (candidate coverage)
- **nDCG@10** (top-ranking quality)

This is the key evidence that fusion (and the chosen fusion method) is justified.

## Hyperparameter sweeps: what we tried and what we learned

Some sweeps are computationally expensive to recompute live in a notebook.

Below we record the key sweeps and outcomes (from `robust04_research_log.md`) to explain the research process.

You can treat these as *experimental evidence* that guided later decisions.


In [ ]:
# Key sweep: DUQGen MonoT5-3B passage reranking depth (top_n)
duqgen_monot5p_topn_sweep = [
    {'model': 'cramraj8/duqgen-monot5-3b-robust04-1k', 'top_n': 200, 'alpha': 0.3, 'map': 0.3456},
    {'model': 'cramraj8/duqgen-monot5-3b-robust04-1k', 'top_n': 300, 'alpha': 0.3, 'map': 0.3514},
    {'model': 'cramraj8/duqgen-monot5-3b-robust04-1k', 'top_n': 500, 'alpha': 0.3, 'map': 0.3649},
    {'model': 'cramraj8/duqgen-monot5-3b-robust04-1k', 'top_n': 800, 'alpha': 0.3, 'map': 0.3727},
    {'model': 'cramraj8/duqgen-monot5-3b-robust04-1k', 'top_n': 1000, 'alpha': 0.3, 'map': 0.3743},
]

print('DUQGen MonoT5-3B passage reranking: top_n sweep (MAP on 50 judged queries)')
for row in duqgen_monot5p_topn_sweep:
    print(row)

# Key later variants that increased passage coverage (best observed, not checkpointed)
coverage_variants = [
    {
        'setting': 'max_passages=10 (stride=1200, doc_max_chars=12000)',
        'map': 0.3759,
    },
    {
        'setting': 'doc_max_chars=20000, max_passages=15 (stride=1200)',
        'map': 0.3767,
    },
]

print('\nCoverage variants (same overall approach, more passage coverage):')
for row in coverage_variants:
    print(row)


In [ ]:
rows = [
    summarize_run_row('minmax fusion run_2', run2, qrels),
    summarize_run_row('RRF run_2', rrf_run2, qrels),
    summarize_run_row('minmax fusion run_3', run3, qrels),
    summarize_run_row('RRF run_3', rrf_run3, qrels),
]
print_table(rows)

print('\nInterpretation (quick talking points):')
print('- If RRF matches or beats min-max: fusion benefit is robust and not dependent on score scales.')
print('- If min-max beats RRF: scores carry useful information after normalization; weighting matters.')
print('- Check Recall@1000: if fusion mainly boosts recall, it strengthens reranking headroom.')

## Methods tried that did not improve (and why they were deprioritized)

Negative results are part of the research process. They help justify why the final pipeline looks the way it does.

Below are approaches we implemented or swept that **did not surpass** the fused baseline (≈0.2997 MAP on judged queries), along with likely reasons.

### 1) TF/TF-IDF/SVD + clustering reranks
What it is:
- Build doc vectors from Lucene term frequencies.
- Transform to TF-IDF and optionally reduce dimension (SVD).
- Cluster docs and use cluster similarity as a reranking signal.

Why it was appealing:
- Cheap, fully unsupervised, can inject “topic structure”.

Why it likely didn’t help:
- It stays close to lexical matching and adds a noisy proxy objective.
- It does not model query-document relevance directly.

### 2) LSH-style hashing over reduced vectors
What it is:
- Convert vectors to binary hashes (mean-threshold or random hyperplanes).

Why it was appealing:
- Extremely fast approximate similarity.

Why it likely didn’t help:
- Hashing is lossy and reduces ranking resolution.

### 3) Dense-embedding hashing / SimHash
What it is:
- Hash dense embeddings and use hash similarity as a reranking feature.

Why it likely didn’t help:
- If you already have a good dense retriever, hashing tends to reduce quality.

### 4) Small cross-encoder rerankers
Why it was appealing:
- Cross-encoders can be strong rerankers.

Why it likely didn’t help:
- Robust04 has long documents; truncation can hide evidence.
- Small models may not have enough capacity; MS MARCO tuning can mismatch Robust04.

### 5) Passage aggregation AvgTopK (vs MaxP)
Observation:
- In our best setting, **MaxP** beat AvgTopK.

Why MaxP makes sense here:
- Robust04 relevance often appears in a single strong span; MaxP captures the strongest evidence passage.

Takeaway:
The approaches that worked best either improved candidate coverage (fusion) or applied a strong relevance model at passage level (MonoT5-3B passage reranking).

## Optional: reproduce heavy reranking runs

The reranking steps can take a long time (especially MonoT5-3B). To avoid accidental long runs, the cells below are guarded by flags.

- Set `RUN_HEAVY_RERANKING = True` to run experiment scripts (if desired).
- Set `RUN_GENERATE_RUN_FILES = True` to generate `run_1.res/run_2.res/run_3.res` via `generate_runs.py`.

This separation keeps the notebook presentation-friendly while preserving reproducibility.

In [ ]:
RUN_HEAVY_RERANKING = False
RUN_GENERATE_RUN_FILES = False

# Output run filenames
OUT1 = 'run_1.res'
OUT2 = 'run_2.res'
OUT3 = 'run_3.res'

# Configure run_3 reranking (matches robust04_final_project.ipynb defaults)
RERANK3_MONOT5_PASSAGES = True
MONOT5P_MODEL = 'zeta-alpha-ai/monot5-3b-inpars-v2-robust04'
MONOT5P_ALPHA = 0.2
MONOT5P_TOP_N = 200
MONOT5P_BATCH_SIZE = 4
MONOT5P_MAX_LENGTH = 512
MONOT5P_DOC_MAX_CHARS = 12000
MONOT5P_PASSAGE_CHARS = 1500
MONOT5P_STRIDE_CHARS = 1200
MONOT5P_MAX_PASSAGES = 8
MONOT5P_AGG = 'max'
MONOT5P_FP16 = True

print('RUN_HEAVY_RERANKING:', RUN_HEAVY_RERANKING)
print('RUN_GENERATE_RUN_FILES:', RUN_GENERATE_RUN_FILES)
print('OUT1/OUT2/OUT3:', OUT1, OUT2, OUT3)
print('RERANK3_MONOT5_PASSAGES:', RERANK3_MONOT5_PASSAGES)

In [ ]:
# Heavy reranking reproduction is delegated to scripts for reproducibility and caching.
# This is the same pipeline used during experiments (passage-level reranking).

if RUN_HEAVY_RERANKING:
    import subprocess
    import sys

    # Example: run the experiment driver on judged queries with extensive passage coverage
    cmd = [
        sys.executable,
        'experiments_cluster_lsh.py',
        '--only-baseline',
    ]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True)
else:
    print('Skipping heavy reranking (set RUN_HEAVY_RERANKING=True to run).')


In [ ]:
# Optional presentation plot for the MonoT5 passage reranking sweep
# (This is lightweight: it only plots the recorded results from the log.)

PLOT_SWEEPS = False

if PLOT_SWEEPS:
    try:
        import matplotlib.pyplot as plt

        xs = [r['top_n'] for r in duqgen_monot5p_topn_sweep]
        ys = [r['map'] for r in duqgen_monot5p_topn_sweep]

        plt.figure(figsize=(6, 3))
        plt.plot(xs, ys, marker='o')
        plt.title('DUQGen MonoT5-3B passage reranking: top_n vs MAP (judged queries)')
        plt.xlabel('top_n reranked')
        plt.ylabel('MAP')
        plt.grid(True, alpha=0.3)
        plt.show()
    except Exception as e:
        print('Could not plot (matplotlib/display not available):', repr(e))
else:
    print('Skipping sweep plot (set PLOT_SWEEPS=True to enable).')

## Final artifacts (what we submit) + how to reproduce

### What we submit
We output three run files in standard TREC 6-column format:

- **`run_1.res`**: RM3 baseline (lexical PRF)
- **`run_2.res`**: fused retrieval (RM3 + SPLADE++ + Dense)
- **`run_3.res`**: fused retrieval (RM3 + SPLADE++ + SPLADE-v3 + Dense) + optional MonoT5 passage reranking

### Why the three-run design makes sense
- `run_1` establishes a strong classical baseline.
- `run_2` shows benefit from mixing lexical + sparse neural + dense neural retrieval.
- `run_3` demonstrates a stronger fusion and (optionally) a second-stage neural reranker for long-document relevance.

### Reproducibility
To generate the run files identically every time (and reuse caching), we call the repo script `generate_runs.py` from this notebook.

The next cell is guarded by `RUN_GENERATE_RUN_FILES` so you don’t accidentally launch long computations.

## Generate run files (`run_1.res`, `run_2.res`, `run_3.res`) from this notebook

This notebook can generate the final submission-format run files by calling the repo script `generate_runs.py`.

Why call the script instead of rewriting logic here?
- It keeps run generation **identical** to the submission notebook (`robust04_final_project.ipynb`).
- It uses disk caching (`/workspace/.cache`) to avoid repeated work.

**Safety:** run generation can take time (and reranking can be heavy), so it only runs if `RUN_GENERATE_RUN_FILES = True`.

## References

- Abdul-Jaleel, N., Allan, J., Croft, W. B., Diaz, F., Larkey, L., Li, X., Smucker, M. D., & Wade, C. (2004). *UMass at TREC 2004: Novelty and Hard*. (RM3-style PRF usage.)
- Cormack, G. V., Clarke, C. L. A., & Buettcher, S. (2009). *Reciprocal Rank Fusion Outperforms Condorcet and Individual Rank Learning Methods*. SIGIR.
- Formal, T., Lassance, C., Piwowarski, B., & Clinchant, S. (2021). *SPLADE: Sparse Lexical and Expansion Model for First Stage Ranking*. SIGIR. (Also arXiv:2107.05720.)
- Järvelin, K., & Kekäläinen, J. (2002). *Cumulated Gain-based Evaluation of IR Techniques*. TOIS. (nDCG)
- Karpukhin, V., Oguz, B., Min, S., Lewis, P., Wu, L., Edunov, S., Chen, D., & Yih, W.-T. (2020). *Dense Passage Retrieval for Open-Domain Question Answering*. EMNLP.
- Lavrenko, V., & Croft, W. B. (2001). *Relevance-Based Language Models*. SIGIR.
- Li, C., Yates, A., MacAvaney, S., He, B., & Sun, Y. (2020). *PARADE: Passage Representation Aggregation for Document Reranking*. arXiv:2008.09093.
- Lin, J., Ma, X., Lin, S.-C., Pradeep, R., & Nogueira, R. (2021). *Pyserini: A Python Toolkit for Reproducible Information Retrieval Research*. SIGIR.
- Malkov, Y. A., & Yashunin, D. A. (2018). *Efficient and robust approximate nearest neighbor search using Hierarchical Navigable Small World graphs*. TPAMI. (Also arXiv:1603.09320.)
- Nogueira, R., Jiang, Z., & Lin, J. (2020). *Document Ranking with a Pretrained Sequence-to-Sequence Model*. arXiv:2003.06713. (MonoT5)
- Robertson, S., & Zaragoza, H. (2009). *The Probabilistic Relevance Framework: BM25 and Beyond*. Foundations and Trends in IR.
- Thakur, N., Reimers, N., Daxenberger, J., & Gurevych, I. (2021). *BEIR: A Heterogeneous Benchmark for Zero-shot Evaluation of Information Retrieval Models*. NeurIPS (Datasets and Benchmarks). arXiv:2104.08663.
- Xiao, S., Liu, Z., Zhang, P., & Muennighoff, N. (2023). *C-Pack: Packaged Resources To Advance General Chinese Embedding*. arXiv:2309.07597. (Introduces BGE embeddings.)

Notes:
- BGE model usage here follows the FlagEmbedding/BAAI releases; we cite the BGE technical report (Xiao et al., 2023) for the embedding family and DPR (Karpukhin et al., 2020) for the dense-retrieval paradigm.

In [ ]:
if RUN_GENERATE_RUN_FILES:
    import subprocess
    import sys

    cmd = [
        sys.executable,
        'generate_runs.py',
        '--out1', OUT1,
        '--out2', OUT2,
        '--out3', OUT3,
    ]

    if RERANK3_MONOT5_PASSAGES:
        cmd += [
            '--rerank3-monot5-passages',
            '--monot5p-model', str(MONOT5P_MODEL),
            '--monot5p-alpha', str(MONOT5P_ALPHA),
            '--monot5p-top-n', str(MONOT5P_TOP_N),
            '--monot5p-batch-size', str(MONOT5P_BATCH_SIZE),
            '--monot5p-max-length', str(MONOT5P_MAX_LENGTH),
            '--monot5p-doc-max-chars', str(MONOT5P_DOC_MAX_CHARS),
            '--monot5p-passage-chars', str(MONOT5P_PASSAGE_CHARS),
            '--monot5p-stride-chars', str(MONOT5P_STRIDE_CHARS),
            '--monot5p-max-passages', str(MONOT5P_MAX_PASSAGES),
            '--monot5p-agg', str(MONOT5P_AGG),
        ]
        if MONOT5P_FP16:
            cmd += ['--monot5p-fp16']

    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True)
else:
    print('Skipping run generation (set RUN_GENERATE_RUN_FILES=True to run).')